# 30. Unsupervised Learning: Apriori Algorithm (Frequent Itemset Mining)

## Algorithm Category
**Type**: Unsupervised Learning - Association Rule Learning  
**Complexity**: Medium  
**Use Case**: Find frequent itemsets and generate association rules from transactional data

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Apriori algorithm and association rule learning
- Implement Apriori for frequent itemset mining
- Generate and evaluate association rules
- Understand support, confidence, and lift metrics
- Apply Apriori to market basket analysis
- Visualize association rules

## Historical Context

Apriori was developed by Agrawal and Srikant in 1994:
- Agrawal, R. & Srikant, R. (1994): "Fast Algorithms for Mining Association Rules"
- Foundation for market basket analysis
- One of the most cited algorithms in data mining

**Key Papers/References:**
- Agrawal, R. & Srikant, R. (1994). "Fast Algorithms for Mining Association Rules"
- Agrawal, R., et al. (1993). "Mining association rules between sets of items in large databases"

## When to Use Apriori Algorithm

Apriori is appropriate when:
- You have transactional data (market basket, clickstream, etc.)
- You want to find frequent patterns
- You need to discover association rules
- Working with categorical/binary data
- Market basket analysis
- Recommendation systems

## Theory & Mechanics

### Mathematical Foundation

Apriori finds frequent itemsets using the downward closure property.

**Key Concepts:**

1. **Support**: Frequency of itemset in transactions
   $$\text{Support}(X) = \frac{|\{t \in T : X \subseteq t\}|}{|T|}$$

2. **Confidence**: Probability of Y given X
   $$\text{Confidence}(X \rightarrow Y) = \frac{\text{Support}(X \cup Y)}{\text{Support}(X)}$$

3. **Lift**: How much more likely Y is when X is present
   $$\text{Lift}(X \rightarrow Y) = \frac{\text{Confidence}(X \rightarrow Y)}{\text{Support}(Y)}$$

**Apriori Principle:**
- If an itemset is frequent, all its subsets are frequent
- If an itemset is infrequent, all its supersets are infrequent

### How It Works

1. **Generate 1-itemsets**: Find frequent single items (above min_support)
2. **Generate k-itemsets**: Join (k-1)-itemsets to form k-itemsets
3. **Prune**: Remove itemsets with infrequent subsets
4. **Count**: Count support for candidate itemsets
5. **Filter**: Keep only frequent itemsets
6. **Repeat**: Steps 2-5 until no more frequent itemsets

### Key Hyperparameters

- **min_support**: Minimum support threshold (0-1)
- **min_confidence**: Minimum confidence for rules (0-1)
- **min_lift**: Minimum lift for rules (typically > 1)

### Advantages

- Simple and intuitive
- Finds meaningful patterns
- Works well with categorical data
- Interpretable results
- Foundation for recommendation systems

### Limitations

- Computationally expensive (exponential in worst case)
- Requires multiple database scans
- Sensitive to min_support threshold
- May generate many rules
- Only finds frequent patterns


## Implementation

Let's implement Apriori for frequent itemset mining and association rules.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("Libraries imported successfully!")


In [ ]:
# Create sample transactional data (market basket)
transactions = [
    ['bread', 'milk'],
    ['bread', 'diaper', 'beer', 'eggs'],
    ['milk', 'diaper', 'beer', 'cola'],
    ['bread', 'milk', 'diaper', 'beer'],
    ['bread', 'milk', 'diaper', 'cola']
]

print("Sample Transactions:")
for i, trans in enumerate(transactions, 1):
    print(f"  Transaction {i}: {trans}")

# Encode transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nEncoded DataFrame Shape: {df.shape}")
print(f"Items: {list(df.columns)}")
print(f"\nEncoded Transactions:")
print(df.head())


In [ ]:
# Find frequent itemsets using Apriori
min_support = 0.4  # 40% minimum support
frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)

print(f"Frequent Itemsets (min_support={min_support}):")
print(frequent_itemsets)

# Add itemset length
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print(f"\nFrequent Itemsets by Length:")
for length in sorted(frequent_itemsets['length'].unique()):
    itemsets = frequent_itemsets[frequent_itemsets['length'] == length]
    print(f"\n  {length}-itemsets ({len(itemsets)} found):")
    for idx, row in itemsets.iterrows():
        items = ', '.join(list(row['itemsets']))
        print(f"    {items}: support = {row['support']:.3f}")


## Association Rules

Let's generate and evaluate association rules.


In [ ]:
# Generate association rules
min_confidence = 0.6
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

print(f"Association Rules (min_confidence={min_confidence}):")
print(f"  Number of rules: {len(rules)}")
print(f"\nRules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# Filter rules with high lift
high_lift_rules = rules[rules['lift'] > 1.0]
print(f"\nRules with Lift > 1.0: {len(high_lift_rules)}")
if len(high_lift_rules) > 0:
    print("\nTop Rules by Lift:")
    top_rules = high_lift_rules.nlargest(5, 'lift')
    for idx, row in top_rules.iterrows():
        antecedents = ', '.join(list(row['antecedents']))
        consequents = ', '.join(list(row['consequents']))
        print(f"  {antecedents} -> {consequents}")
        print(f"    Support: {row['support']:.3f}, Confidence: {row['confidence']:.3f}, Lift: {row['lift']:.3f}")


## Effect of Support Threshold

Let's see how min_support affects the number of frequent itemsets.


In [ ]:
# Test different support thresholds
support_thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
results = []

for min_sup in support_thresholds:
    itemsets = apriori(df, min_support=min_sup, use_colnames=True)
    num_itemsets = len(itemsets)
    results.append({
        'support': min_sup,
        'num_itemsets': num_itemsets
    })
    print(f"min_support={min_sup}: {num_itemsets} frequent itemsets")

# Visualize
plt.figure(figsize=(10, 6))
supports = [r['support'] for r in results]
counts = [r['num_itemsets'] for r in results]
plt.plot(supports, counts, 'o-', markersize=8)
plt.xlabel('Minimum Support')
plt.ylabel('Number of Frequent Itemsets')
plt.title('Effect of Support Threshold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the rules and check metrics.


In [ ]:
# Validate rules
print("Rule Validation:")
print(f"  Total rules generated: {len(rules)}")
print(f"  Rules with lift > 1: {len(rules[rules['lift'] > 1])}")
print(f"  Rules with lift > 1.2: {len(rules[rules['lift'] > 1.2])}")

# Check rule metrics
print(f"\nRule Statistics:")
print(f"  Average support: {rules['support'].mean():.3f}")
print(f"  Average confidence: {rules['confidence'].mean():.3f}")
print(f"  Average lift: {rules['lift'].mean():.3f}")

# Assertions
assert len(frequent_itemsets) > 0, "Should find at least one frequent itemset"
assert len(rules) > 0, "Should generate at least one rule"
assert rules['confidence'].min() >= min_confidence, "All rules should meet min_confidence"
print("\n✓ Validation checks passed")


## Real-World Application

Let's create a more realistic market basket analysis example.


In [ ]:
# Create larger transactional dataset
np.random.seed(42)
items = ['bread', 'milk', 'butter', 'cheese', 'eggs', 'yogurt', 'chicken', 'beef', 
         'apple', 'banana', 'orange', 'tomato', 'lettuce', 'onion', 'potato']

# Generate random transactions
num_transactions = 100
transactions_large = []
for _ in range(num_transactions):
    num_items = np.random.randint(2, 6)
    transaction = np.random.choice(items, size=num_items, replace=False).tolist()
    transactions_large.append(transaction)

# Encode
te_large = TransactionEncoder()
te_ary_large = te_large.fit(transactions_large).transform(transactions_large)
df_large = pd.DataFrame(te_ary_large, columns=te_large.columns_)

print(f"Large Dataset:")
print(f"  Number of transactions: {len(transactions_large)}")
print(f"  Number of items: {len(items)}")
print(f"  Average items per transaction: {df_large.sum(axis=1).mean():.1f}")

# Find frequent itemsets
frequent_itemsets_large = apriori(df_large, min_support=0.1, use_colnames=True)
frequent_itemsets_large['length'] = frequent_itemsets_large['itemsets'].apply(lambda x: len(x))

print(f"\nFrequent Itemsets (min_support=0.1):")
print(f"  Total: {len(frequent_itemsets_large)}")
print(f"  By length:")
for length in sorted(frequent_itemsets_large['length'].unique()):
    count = len(frequent_itemsets_large[frequent_itemsets_large['length'] == length])
    print(f"    {length}-itemsets: {count}")

# Generate rules
rules_large = association_rules(frequent_itemsets_large, metric="confidence", min_threshold=0.5)
print(f"\nAssociation Rules (min_confidence=0.5):")
print(f"  Total rules: {len(rules_large)}")

# Show top rules by lift
if len(rules_large) > 0:
    top_rules_large = rules_large.nlargest(10, 'lift')
    print(f"\nTop 10 Rules by Lift:")
    for idx, row in top_rules_large.iterrows():
        antecedents = ', '.join(list(row['antecedents']))
        consequents = ', '.join(list(row['consequents']))
        print(f"  {antecedents} -> {consequents}")
        print(f"    Support: {row['support']:.3f}, Confidence: {row['confidence']:.3f}, Lift: {row['lift']:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Apriori Basics**
   - Frequent itemset mining algorithm
   - Uses downward closure property (Apriori principle)
   - Generates association rules
   - Foundation for market basket analysis

2. **Key Metrics**
   - **Support**: Frequency of itemset
   - **Confidence**: Probability of consequent given antecedent
   - **Lift**: How much more likely consequent is with antecedent
   - **Lift > 1**: Positive association
   - **Lift < 1**: Negative association

3. **Algorithm Steps**
   - Generate 1-itemsets
   - Join to form k-itemsets
   - Prune infrequent itemsets
   - Count support
   - Repeat until no more frequent itemsets

4. **Best Practices**
   - Choose appropriate min_support (not too low, not too high)
   - Use min_confidence to filter weak rules
   - Focus on rules with lift > 1
   - Consider business context when interpreting rules
   - Visualize rules for better understanding

### When to Use Apriori Algorithm

✅ **Good for:**
- Market basket analysis
- Transactional data
- Finding frequent patterns
- Association rule mining
- Recommendation systems
- Cross-selling strategies
- Categorical/binary data

❌ **Not ideal for:**
- Continuous/numerical data
- Very large datasets (computational cost)
- When patterns are rare (low support)
- Real-time applications (slow)
- When you need exact counts (approximate methods faster)

### Next Steps

- Try **FP-Growth** algorithm (faster alternative)
- Use **Eclat** algorithm (vertical data format)
- Apply to **recommendation systems**
- Explore **sequential pattern mining**
- Use for **cross-selling** in e-commerce
